In [ ]:
%pip install -q pypdf faiss-cpu ollama ipywidgets

In [ ]:
from io import BytesIO
from pypdf import PdfReader
from ipywidgets import widgets
from IPython.display import display
import ollama
import faiss
import numpy as np

EMBED_MODEL = "qwen3-embedding:latest"
CHAT_MODEL = "qwen3:1.7b"

print("All library are installed successfully")

In [ ]:
def load_and_chunk_pdf(pdf_bytes, chunk_size=1000, overlap=150):
    reader = PdfReader(BytesIO(pdf_bytes))

    full_text = ""
    for page in reader.pages:
        full_text += page.extract_text() + "\n"

    chunks = []
    start = 0

    while start < len(full_text):
        end = start + chunk_size
        chunk = full_text[start:end]
        chunks.append(chunk)
        start += chunk_size - overlap

    return chunks

In [ ]:
uploader = widgets.FileUpload(accept='.pdf', multiple=False, description="Upload PDF") 
display(uploader)

In [ ]:
uploader = widgets.FileUpload(accept='.pdf', multiple=False, description="Upload PDF") 
display(uploader)

In [ ]:
import time
EMBED_MODEL = "qwen3-embedding:latest"
CHAT_MODEL = "qwen3:1.7b"

print("Embedding Model:", EMBED_MODEL)
print("Chat Model:", CHAT_MODEL)


def embed_chunks(texts, batch_size=2, retries=3):
    all_embeddings = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]

        for attempt in range(retries):
            try:
                response = ollama.embed(model=EMBED_MODEL,input=batch)
                all_embeddings.extend(response["embeddings"])
                break

            except Exception as e:
                print(f"Error embedding batch {i // batch_size + 1}: {e}")
                if attempt < retries - 1:
                    time.sleep(2)
                else:
                    raise

    return all_embeddings

chunk_embeddings = embed_chunks(pdf_chunks)
chunk_embeddings = np.array(chunk_embeddings, dtype="float32")

print("Embeddings generated:", chunk_embeddings.shape)
dimension = chunk_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(chunk_embeddings)

print("FAISS index created. Total vectors:", index.ntotal)

In [ ]:
def retrive_relevant_chunks(query, top_k=8):
    query_embedding = embed_chunks([query])

    distance, indices = index.search(np.array(query_embedding).astype("float32"),top_k)
    relevant_chunks = [pdf_chunks[i]for i in indices[0]]

    return relevant_chunks


text_question = "What is this document about?"

found_chunks = retrive_relevant_chunks(text_question)

print("Found relevant chunks for the question:", text_question)

for i, chunk in enumerate(found_chunks, 1):
    print(f"Chunk {i} preview:", chunk[:200], "...") # print first 200 character of each releveant chunk


In [ ]:
def ask_pdf(question, top_k=8):
    relevant_chunks = retrive_relevant_chunks(question, top_k)
    context = "\n\n---\n\n".join(relevant_chunks)

    prompt = f"""You are answering questions about a document using the excerpts below. Read all the excerpts carefully; the answer may be spread across multiple excerpts.

Excerpts:
{context}

Question:
{question}

Answer:
"""

    response = ollama.chat(
        model=CHAT_MODEL,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        options={"num_ctx": 4096}
    )

    return response["message"]["content"]


answer = ask_pdf("What is this document about?")
print("Answer:", answer)

In [ ]:
question_box = widgets.Text(
    description="Enter your question:",
    placeholder="Type your question here...",
    layout=widgets.Layout(width="80%"),
    continuous_update=False
)

ask_button = widgets.Button(description="Ask PDF",button_style="primary")
chat_output = widgets.Output()


def handle_questions(_):
    question = question_box.value.strip()

    if not question:
        return
    question_box.value = ""

    with chat_output:
        print(f"Question: {question}")
        answer = ask_pdf(question)
        print(f"Answer: {answer}")

ask_button.on_click(handle_questions)
question_box.observe(handle_questions, names="value")

display(widgets.HBox([question_box, ask_button]),chat_output)